### token-level or seq-level

- Token 级别优势估计：这类方法为序列中的每一步（Token）计算优势 $A_t$，通常依赖于价值函数 $V_t$ 或者蒙特卡洛回报 $G_t$（对于序列中的每一个 token，都计算一个独立的、可能不同的优势值。这意味着优势函数考虑了每个时间步（token）的即时奖励（reward）和未来价值（value）。最终的 advantages 张量 (batch_size, seq_len) 中，在 seq_len 维度上的值是不同的。）
    - GAE
    - REINFORCE++ (RF++)
    - REMAX
- 序列级别（结果导向）优势估计 ：首先为整个序列计算一个单一的标量奖励或分数（通常是对 token_level_rewards 求和），然后基于这个序列总分计算一个优势值。这个单一的优势值会被广播（broadcast）到该序列中的每一个 token 上。因此，对于同一个样本，其 advantages 张量 (batch_size, seq_len) 在 seq_len 维度上的值是相同的（除了被 mask 的部分）。这类算法通常只关心最终结果的好坏，而不是中间过程。
    - GRPO
    - RLOO (Reinforcement Learning with Leave-One-Out)
    - REINFORCE++-Baseline
- 最终返回的 shape 是一致的（seq-level 通过广播实现, `(B, ) => (B, L)`）

### 批次标准化（batch whitening）与组内标准化 （group）

> 在强化学习（RL），特别是策略梯度算法（如 PPO 及其变体）中，对优势（Advantage）进行归一化处理是稳定训练、降低梯度方差的关键步骤。其目的通常是调整优势值的尺度，使其分布更易于学习。
- “批次标准化”和“组内标准化”是两种核心策略。它们的核心区别在于计算归一化所需的统计量（均值和标准差）时所使用的数据范围（Scope）。
    - 批次（Batch）：指一次训练迭代中使用的全部数据样本。
    - 组（Group）：指批次中具有共同输入（例如，同一个 Prompt $q$）的所有响应样本集合。

| 样本 ID | 组 (Prompt) | 奖励 (R) |
|---|---|---|
| 1 | A (简单任务) | 10 |
| 2 | A (简单任务) | 8 |
| 3 | B (困难任务) | 2 |
| 4 | B (困难任务) | 0 |

- 组内归一化
    - $\mu_A=9,\sigma_A=1;\mu_B=1,\sigma_B=1$
    - 标准化后优势:
        - 样本1：(10-9)/1=1
        - 样本2：(8-9)/1=-1
        - 样本3：(2-1)/1=1
        - 样本4：(0-1)/1=-1
- 批次归一化
    - $\mu_{Batch}=5,\sigma_{Batch}=4.12$
- 批次标准化：强调全局表现。优势值表示“该响应相对于批次中所有响应（无论来自哪个 Prompt）的平均水平有多好”。
    - 组内标准化：强调局部排序（Intra-prompt Ranking）。优势值表示“该响应相对于同一个 Prompt 的其他响应有多好”。这对于需要从多个候选中选择最佳响应的任务（如 RLHF 中的偏好学习）尤其重要。
- 基线（Baseline）的定义: 在策略梯度中，中心化相当于引入了一个基线来降低方差。
    - 组内中心化：使用的是一个依赖于输入状态（Prompt）的局部基线（$\mu_x$）。这种状态相关的基线通常比全局基线能更有效地降低方差。
    - 批次中心化：使用的是一个全局基线（$\mu_{Batch}$）。

### GAE

$$
\begin{split}
&\delta_t=r_t+\gamma V_\phi(s_{t+1})-V_\phi(s_t)\\
&A_t^{\text{GAE}(\gamma,\lambda)}=\sum_{k=t}^{T-1}(\gamma\lambda)^{k-t}\delta_k
\end{split}
$$

- 代码的实现
    - $A_t=\delta_t+\gamma\lambda A_{t+1}$ （`lastgaelam = delta + gamma * lam * lastgaelam`）
    - $G_t=A_t+V(s_t)$（这里相当于 V_old）
        - 用来训练价值网络：$L_{VF}=\frac{1}N\sum (V(s_t)-G_t)^2$（更新 V_current）
            - `compute_value_loss`
            - VF：value function
            - $\lambda = 0$，$L_{vf}=\frac{1}N\sum (r_t+\gamma V_\phi(s_{t+1})-V_\phi(s_t))^2$ (Actor-critic)
- GAE vs. REINFORCE++
    - 用什么作为基线，
        - GAE 用一个单独的价值函数 $V(s_t)$，
        - REINFORCE++：从批次回报数据计算统计基线（所谓的白化），$A_t=\frac{G_t-\mu_G}{\sigma_G+\epsilon}$

### REINFORCE++, REINFORCE++-bl

- RF++
    - 不依赖分组，它直接使用蒙特卡洛回报（discounted future returns）作为优势，并对其进行 whiten 处理。
    - 时间步 $t$，回报 $G_t=\sum_{k=t}^{T-1}\gamma^{k-t}r_k$
    - $\hat A_t=\text{whiten}(G_t)=\frac{G_t-\mu_G}{\sigma_G+\epsilon}$
- RF++-bl
    - https://mp.weixin.qq.com/s/BWUUoR2BCkauGFvamiExZA
        - 将 PPO 的 Critic Network 替换为 group reward mean，然后使用全局优势归一化，并使用无偏的K2 KL估计器来计算KL损失。
            - http://joschu.net/blog/kl-approx.html
            - $k_2=\frac12\left(\log\frac{p(x)}{q(x)}\right)^2$ ($q(x)=\pi_\theta, p(x)=\pi_{ref}$)
    - 该算法与 GRPO 非常相似，也使用组内平均奖励作为基线。主要区别在于计算出 (序列奖励 - 组均值) 后，会进行 whiten 操作

In [1]:
import numpy as np

In [3]:
token_level_rewards = np.zeros((5, 8))

In [6]:
token_level_rewards[:, -1] = np.random.randint(0, 2, 5)

In [7]:
token_level_rewards

array([[0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1.]])

In [14]:
gamma = 0.9
running_return = 0
returns = np.zeros_like(token_level_rewards)
response_mask = np.ones_like(token_level_rewards)
for t in reversed(range(token_level_rewards.shape[1])):
    running_return = token_level_rewards[:, t] + gamma * running_return
    returns[:, t] = running_return
    # Reset after EOS
    running_return = running_return * response_mask[:, t]
returns

array([[0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ]])

### RLOO (vs. GRPO)

- GRPO:
    - $A_i^\text{GRPO}=s_i-\mu_g$，其中 $\mu_g=\frac1N\sum s_i$
    - $A_i^\text{GRPO-Norm}=\frac{s_i-\mu_g}{\sigma_G+\epsilon}$
- RLOO: (N: group size)
    - $\mu_{g \backslash {i}}=\frac1{N-1}\sum_{j\neq i}^Ns_j$
    - $A_{i}^{\text{RLOO}}=s_i-\mu_{g\backslash {i}}=s_i-\frac{\sum s_j-s_i}{N-1}=\frac{(N-1)s_i-\sum s_j+s_i}{N-1}=\frac{Ns_i-\sum \mu_g}{N-1}=\frac{Ns_i-N\mu_g}{N-1}=(s_i-\mu_g)\frac{N}{N-1}$

### misc

In [15]:
import torch

bs = 4  
response_length = 5 

scores = torch.tensor([10.0, 20.0, 30.0, 40.0])  # shape: (bs,) -> (4,)

response_mask = torch.tensor([
    [1, 1, 1, 0, 0],
    [1, 1, 1, 1, 0],
    [1, 1, 0, 0, 0],
    [1, 1, 1, 1, 1],
], dtype=torch.float32)  # shape: (bs, response_length) -> (4, 5)

In [16]:
scores.unsqueeze(-1) * response_mask

tensor([[10., 10., 10.,  0.,  0.],
        [20., 20., 20., 20.,  0.],
        [30., 30.,  0.,  0.,  0.],
        [40., 40., 40., 40., 40.]])

In [17]:
scores.unsqueeze(-1).tile([1, response_length]) * response_mask

tensor([[10., 10., 10.,  0.,  0.],
        [20., 20., 20., 20.,  0.],
        [30., 30.,  0.,  0.,  0.],
        [40., 40., 40., 40., 40.]])